## For Dimension Table Artists

In [1]:
import pandas as pd

df_temporal = pd.read_csv('../data/processed/clean_data/temporal_artists.csv', index_col=0).reset_index()
df_features = pd.read_csv('../data/processed/clean_data/lastfm_features.csv', index_col=0).reset_index()

dim_artists = pd.merge(
    df_temporal,
    df_features,
    left_on='primary_artist_name',
    right_on='artist',
    how='left'
)

dim_artists = dim_artists.drop(columns=['artist', 'index'])
dim_artists['tags'] = dim_artists['tags'].fillna('')
dim_artists['similar_artists'] = dim_artists['similar_artists'].fillna('')

dim_artists.insert(0, 'artists_id', range(1, len(dim_artists) + 1))

dim_artists

,artists_id,primary_artist_name,artist_listeners,artist_playcount,tags,similar_artists,num_tags,num_similar_artists
0,1,(Hed) P.E.,150657,4410090,"rapcore, Nu Metal, crossover, g-punk, alternat...","(hed) Planet Earth, Reveille, Primer 55, Coal ...",10,20
1,2,*NSYNC,2157048,32982790,"pop, boybands, 90s, dance, boy band, american,...","JC Chasez, Backstreet Boys, O-Town, Justin Tim...",10,20
2,3,2 Chainz,1490344,27551890,"Hip-Hop, rap, trap, Dirty South, southern rap,...","Young Jeezy, Lil' Wayne, Gucci Mane, T.I., Ric...",10,20
3,4,21 Savage,2407273,227824623,"trap, Hip-Hop, rap, hip hop, atlanta, british,...","Metro Boomin, Future, Offset, Travi$ Scott, Li...",10,20
4,5,2Pac,4312766,153016025,"rap, Hip-Hop, Gangsta Rap, West Coast Rap, hip...","Thug Life, The Notorious B.I.G., Dr. Dre, Snoo...",10,20
...,...,...,...,...,...,...,...,...
588,589,wave to earth,937000,96935028,"indie pop, Korean, k-indie, indie rock, no ski...","Apro, Kim Daniel, Potatoi, LEEHEESANG, WIMY, H...",10,20
589,590,will.i.am,2669817,31468310,"Hip-Hop, rap, rnb, funk, hip hop, pop, america...","Black Eyed Peas, Flo Rida, Taio Cruz, Pitbull,...",10,20
590,591,yuji,182575,2617801,"malaysian, indie, rnb, ost, video","yung kai, Aisha Retno, Noh Salleh, Insomniacks...",5,20
591,592,¥$,1551572,85258501,"Hip-Hop, rap, Kanye West, hip hop, trap, pop r...","Kanye West, DONDA, KIDS SEE GHOSTS, King Combs...",10,20


## Dimension Table: Tracks

In [2]:
saved_tracks = pd.read_csv('../data/processed/clean_data/saved_tracks.csv', index_col=0).reset_index()
recently_played = pd.read_csv('../data/processed/clean_data/recently_played.csv', index_col=0).reset_index()
candidate_tracks = pd.read_csv('../data/processed/clean_data/candidate_tracks.csv', index_col=0).reset_index()

saved_tracks['release_date'] = pd.to_datetime(saved_tracks['release_date'], errors='coerce', format='mixed')

"""
    saved_tracks cols: track_id, album_name, release_date, song_name, primary_artist_name, duration_min
    recently_played cols: track_id, album_name, release_date, song_name, primary_artist_name, duration_min
    candidate_tracks: track_id, album_name, release_date, song_name, primary_artist_name, duration_min
"""

saved_tracks = saved_tracks.rename(columns={'id' : 'track_id'})
recently_played = recently_played.rename(columns={'id' : 'track_id'})
candidate_tracks = candidate_tracks.rename(columns={'id' : 'track_id',
                                                    'name' : 'song_name'})

candidate_tracks['release_date'] = pd.to_datetime(candidate_tracks['release_date'], errors='coerce', format='mixed')

cols = ['track_id', 'release_date', 'song_name', 'album_name','primary_artist_name', 'duration_min']

saved_tracks = saved_tracks[cols]
recently_played = recently_played[cols]
candidate_tracks = candidate_tracks[cols]

dim_tracks = pd.concat([saved_tracks, recently_played, candidate_tracks], ignore_index=True)
dim_tracks = dim_tracks.drop_duplicates(subset=['track_id']).reset_index(drop=True)

dim_tracks

,track_id,release_date,song_name,album_name,primary_artist_name,duration_min
0,2EoOZnxNgtmZaD8uUmz2nD,1994-03-09 00:00:00,Black Hole Sun,Superunknown (Deluxe Edition),Soundgarden,5.31
1,0NfYAsKygCYwPA2BgTZ1qg,2012-10-22 00:00:00,Now Or Never - Bonus Track,"good kid, m.A.A.d city (Deluxe)",Kendrick Lamar,4.27
2,3tYxhPqkioZEV5el3DJxLQ,2013-04-05 00:00:00,Youngest Daughter,Jar,Superheaven,4.16
3,4PEGwWH4tL6H7dGl4uVSPg,1992-10-26 00:00:00,Like a Tattoo,Love Deluxe,Sade,3.63
4,2OM0MeY7mPIRbs8SCzRdTH,2024-02-16 00:00:00,Familia,2093,Yeat,2.58
...,...,...,...,...,...,...
4681,1AsQFm7ORbr4NEZ0dUltOY,2017-02-24 00:00:00,Way It Goes,Landmark,Hippo Campus,4.09
4682,45ewhNby8MgyWw6HmT7HKJ,2018-09-28 00:00:00,Bambi,Bambi,Hippo Campus,3.24
4683,3hWfXpVhqsALMNxzGcCxeI,2017-09-12 00:00:00,Warm Glow,Warm Glow,Hippo Campus,5.98
4684,7GUPYx7ywjcqBIUXssYjaW,2017-02-24 00:00:00,Buttercup,Landmark,Hippo Campus,3.78


## Fact Table Interactions

In [3]:
saved_tracks['interaction_type'] = 'Saved'
recently_played['interaction_type'] = 'Recently Played'

long_top_tracks = pd.read_csv('../data/processed/clean_data/long_top_tracks.csv', index_col=0).reset_index()
long_top_tracks = long_top_tracks.rename(columns={'id' : 'track_id',
                                                'name' : 'song_name'})

long_top_tracks['interaction_type'] = 'Long Top Track'

necessary_cols = ['track_id', 'interaction_type']

st = saved_tracks[necessary_cols]
rp = recently_played[necessary_cols]
lt = long_top_tracks[necessary_cols]

fact_interactions = pd.concat([st, rp, lt], ignore_index=True)
fact_interactions.insert(0, 'interaction_id',range(1, len(fact_interactions) + 1))

fact_interactions

,interaction_id,track_id,interaction_type
0,1,2EoOZnxNgtmZaD8uUmz2nD,Saved
1,2,0NfYAsKygCYwPA2BgTZ1qg,Saved
2,3,3tYxhPqkioZEV5el3DJxLQ,Saved
3,4,4PEGwWH4tL6H7dGl4uVSPg,Saved
4,5,2OM0MeY7mPIRbs8SCzRdTH,Saved
...,...,...,...
2506,2507,2JuasWPUodaUxf5nwNpciQ,Long Top Track
2507,2508,3UBxPhstwOkNG2YZ1N2XhQ,Long Top Track
2508,2509,5L0KDoZklMgs9GPoonneEl,Long Top Track
2509,2510,7CZyCXKG6d5ALeq41sLzbw,Long Top Track


In [4]:
dim_tracks.columns

Index(['track_id', 'release_date', 'song_name', 'album_name',
       'primary_artist_name', 'duration_min'],
      dtype='str')

In [5]:
dim_artists.columns

Index(['artists_id', 'primary_artist_name', 'artist_listeners',
       'artist_playcount', 'tags', 'similar_artists', 'num_tags',
       'num_similar_artists'],
      dtype='str')

In [6]:
fact_interactions.columns

Index(['interaction_id', 'track_id', 'interaction_type'], dtype='str')

In [7]:
artist_id_map = dim_artists[['artists_id', 'primary_artist_name']]
dim_tracks = dim_tracks.merge(artist_id_map, on='primary_artist_name', how='left')
dim_tracks = dim_tracks.drop(columns=['primary_artist_name'])

dim_tracks = dim_tracks.dropna(subset=['artists_id'])
dim_tracks['artists_id'] = dim_tracks['artists_id'].astype(int)

dim_tracks

,track_id,release_date,song_name,album_name,duration_min,artists_id
0,2EoOZnxNgtmZaD8uUmz2nD,1994-03-09 00:00:00,Black Hole Sun,Superunknown (Deluxe Edition),5.31,449
1,0NfYAsKygCYwPA2BgTZ1qg,2012-10-22 00:00:00,Now Or Never - Bonus Track,"good kid, m.A.A.d city (Deluxe)",4.27,238
2,3tYxhPqkioZEV5el3DJxLQ,2013-04-05 00:00:00,Youngest Daughter,Jar,4.16,465
3,4PEGwWH4tL6H7dGl4uVSPg,1992-10-26 00:00:00,Like a Tattoo,Love Deluxe,3.63,413
4,2OM0MeY7mPIRbs8SCzRdTH,2024-02-16 00:00:00,Familia,2093,2.58,559
...,...,...,...,...,...,...
4606,4kyKnNbxFe86MMeV4xpDL8,2005-01-01 00:00:00,The Thrill Is Gone,B.B. King & Friends - 80,5.02,38
4607,1bursm5ZnNHQS55a23h7kR,1997-11-04 00:00:00,Rock Me Baby,Deuces Wild,6.62,38
4642,4ufiwgOtX1ceUsuTdcKK5f,2005-09-19 00:00:00,Stardust,The Essential Frank Sinatra with the Tommy Dor...,3.19,162
4643,6foCVw9BzZrRSc7Dasw7xA,2005-09-19 00:00:00,I'll Never Smile Again,The Essential Frank Sinatra with the Tommy Dor...,3.17,162


In [8]:
filenames = ['dim_tracks', 'dim_artists', 'fact_interactions']
files = [dim_tracks, dim_artists, fact_interactions]

for file, fn in zip(files, filenames):
    file.to_csv(f'../data/processed/to_db/{fn}.csv', index=False)

In [9]:
dim_tracks = pd.read_csv('../data/processed/to_db/dim_tracks.csv')
fact_interactions = pd.read_csv('../data/processed/to_db/fact_interactions.csv')

valid_facts = fact_interactions[fact_interactions['track_id'].isin(dim_tracks['track_id'])]

print(f"Original interactions: {len(fact_interactions)}")
print(f"Valid interactions: {len(valid_facts)}")
print(f"Dropped {len(fact_interactions) - len(valid_facts)} orphaned rows.")

valid_facts.to_csv('../data/processed/to_db/fact_interactions.csv', index=False)

Original interactions: 2511
Valid interactions: 866
Dropped 1645 orphaned rows.


## Backfilled Candidate Features Data

In [18]:
candidate_tracks = pd.read_csv('../data/processed/clean_data/candidate_tracks.csv')
backfilled_candidates = pd.read_csv('../data/processed/clean_data/candidate_artists_lastfm.csv')

candidate_inference = pd.merge(
    candidate_tracks,
    backfilled_candidates,
    on='primary_artist_name',
    how='left'
)

candidate_inference['tags'] = candidate_inference['tags'].fillna('')
candidate_inference['similar_artists'] = candidate_inference['similar_artists'].fillna('')
candidate_inference['release_date'] = pd.to_datetime(candidate_inference['release_date'], errors='coerce',
                                                     format='mixed')

candidate_inference = candidate_inference.drop('Unnamed: 0', axis=1)
candidate_inference.to_csv('../data/processed/clean_data/full_candidate_data.csv', index=False)